In [13]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_openrouter import ChatOpenRouter
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated
import operator

In [2]:
load_dotenv()

False

In [ ]:
model = ChatOpenAI(model='gpt-4o-mini')

In [5]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [9]:
essay = '''
    Artificial Intelligence (AI) is changing the world, and Pakistan is also becoming part of this technological revolution. AI refers to computer systems that can perform tasks that usually require human intelligence, such as learning, problem-solving, recognizing images, and understanding language. Today, AI is being used in education, healthcare, agriculture, banking, and many other fields. Pakistan has started taking important steps to adopt AI and prepare its youth for the future.

One of the biggest advantages of AI in Pakistan is its ability to improve education. Students can use AI-powered tools to learn difficult subjects, practice languages, and receive personalized guidance. Teachers can also use AI to prepare lessons and check assignments more efficiently. This makes learning faster, easier, and more interactive.

In the healthcare sector, AI is helping doctors diagnose diseases more accurately and quickly. Hospitals can use AI to analyze medical reports, detect illnesses at an early stage, and improve patient care. In agriculture, AI helps farmers monitor crops, predict weather conditions, and use water and fertilizers more efficiently. These improvements can increase food production and reduce waste.

Pakistan's growing IT industry is also benefiting from AI. Many software companies and freelancers are using AI to develop smarter applications, improve customer service, and automate repetitive tasks. The government has also introduced AI-related policies and training programs to develop digital skills among young people and encourage innovation.

However, the AI revolution also brings challenges. Many traditional jobs may change as automation becomes more common. There are also concerns about data privacy, cybersecurity, and the ethical use of AI. To overcome these challenges, Pakistan must invest in quality education, research, digital infrastructure, and responsible AI policies.

In conclusion, the AI revolution offers Pakistan a great opportunity to achieve economic growth and technological advancement. If the government, educational institutions, and private sector work together, AI can improve people's lives, create new jobs, and strengthen the country's economy. By preparing its youth with modern skills, Pakistan can become an active participant in the global AI revolution rather than simply following it.
'''

In [ ]:
prompt = f'Evaluation the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt).score
structured_model.invoke(prompt).feedback

In [14]:
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str

    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [15]:
def evaluate_language(state: UPSCState):

    prompt = f'Evaluation the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [16]:
def evaluate_analysis(state: UPSCState):

    prompt = f'Evaluation the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [17]:
def evaluate_thought(state: UPSCState):

    prompt = f'Evaluation the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'thought_feedback': output.feedback, 'individual_scores': [output.score]}

In [18]:
def final_evaluation(state: UPSCState):

    prompt = f'Based on the follwoing feedbacks breate a summarized feedback \n language feedback - {state['language_feedback']} \n depth of analysis feedback - {state['analysis_feedback']} \n clarity of thought feedback - {state['clarity_feedback']}'
    overall_feedback = model.invoke(prompt).content

    avg_scores = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_scores': avg_scores}

In [25]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaulate_analysis', evaluate_analysis)
graph.add_node('evaulate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)


# edges

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaulate_analysis')
graph.add_edge(START,'evaulate_thought')
graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evaulate_analysis','final_evaluation')
graph.add_edge('evaulate_thought','final_evaluation')
graph.add_edge('final_evaluation', END)

workflow = graph.compile()


In [ ]:
initial_state = {
    'essay': essay
}
final_state = workflow.invoke(initial_state)